# 04 - Backtest Analysis

Walk-forward backtest of the flagship **multi-asset rotation** strategy, a full **tearsheet**, the **Deflated Sharpe Ratio**, and **BHY multiple-testing** correction across strategy variants.

In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
# hmmlearn reports non-convergence via logging (not warnings), so it
# bypasses the filter above; quiet it for clean notebook output.
import logging

logging.getLogger("hmmlearn").setLevel(logging.ERROR)
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the quantcortex repo root on the path (notebooks live in research/).
for p in ("..", "."):
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.insert(0, ap)

RNG = np.random.default_rng(42)

def load_prices(symbols, start="2018-01-01", periods=1800):
    """Real prices via yfinance if available, else deterministic synthetic GBM."""
    try:
        from data.providers.yfinance_provider import YFinanceProvider
        px = YFinanceProvider().get_prices(list(symbols), start=start)
        if px is not None and not px.empty and px.shape[0] > 120:
            return px.dropna(how="all").ffill().dropna()
        raise RuntimeError("empty frame")
    except Exception as exc:
        print(f"[offline] yfinance unavailable ({type(exc).__name__}); using synthetic data.")
    dates = pd.bdate_range(start, periods=periods)
    mu = RNG.normal(0.0003, 0.00015, len(symbols))
    vol = RNG.uniform(0.008, 0.018, len(symbols))
    shocks = RNG.normal(mu, vol, size=(periods, len(symbols)))
    return pd.DataFrame(100 * np.exp(np.cumsum(shocks, axis=0)),
                        index=dates, columns=list(symbols))

def synth_ohlcv(close):
    """Build an OHLCV frame around a close series (synthetic intrabar range)."""
    close = close.dropna()
    hi = close * (1 + np.abs(RNG.normal(0, 0.004, len(close))))
    lo = close * (1 - np.abs(RNG.normal(0, 0.004, len(close))))
    op = close.shift(1).fillna(close.iloc[0])
    vol = RNG.integers(1_000_000, 6_000_000, len(close)).astype(float)
    return pd.DataFrame({"open": op, "high": hi, "low": lo, "close": close, "volume": vol})

print("quantcortex research environment ready.")


quantcortex research environment ready.


In [2]:
symbols = ["QQQ","VGT","GLD","TLT","SPY","VIG"]
prices = load_prices(symbols)
print("rotation universe:", list(prices.columns), prices.shape)

rotation universe: ['QQQ', 'VGT', 'GLD', 'TLT', 'SPY', 'VIG'] (2123, 6)


## Generate weekly target weights

In [3]:
from strategies.multi_asset_rotation import MultiAssetRotation

weekly = prices.index[prices.index.weekday == 0]      # Mondays
strat = MultiAssetRotation()
W = strat.generate_weights(prices, weekly)
print("weekly target-weight panel:", W.shape)
assert bool((W.sum(axis=1) <= 1.0 + 1e-6).all())
W.tail(4).round(3)

weekly target-weight panel: (397, 6)


,QQQ,VGT,GLD,TLT,SPY,VIG
date,,,,,,
2026-05-11,0.0,0.0,0.841,0.159,0.0,0.0
2026-05-18,0.0,0.0,0.372,0.128,0.0,0.0
2026-06-01,0.0,0.0,0.489,0.011,0.0,0.0
2026-06-08,0.0,0.0,0.000,0.000,0.0,0.0


## Run the backtest with transaction costs

In [4]:
from backtest.costs.transaction_costs import TransactionCostModel
from backtest.engines.vectorized import VectorizedBacktest

result = VectorizedBacktest(TransactionCostModel()).run(W, prices)
rets = result.returns.dropna()
print(result.summary())

{'total_return': 0.01622902768979828, 'cagr': 0.0019127472100843868, 'ann_vol': np.float64(0.08788793767588826), 'sharpe': 0.06576267234620249, 'max_drawdown': -0.17919578299874506}


## Tearsheet

In [5]:
from backtest.metrics.tearsheet import Tearsheet

ts = Tearsheet(rets)
metrics = ts.compute()
for k in ["cagr","ann_vol","sharpe","sortino","calmar","max_drawdown","var_95","cvar_95"]:
    print(f"{k:>14}: {metrics[k]:+.4f}")

          cagr: +0.0019
       ann_vol: +0.0879
        sharpe: +0.0658
       sortino: +0.0893
        calmar: +0.0107
  max_drawdown: -0.1792
        var_95: +0.0084
       cvar_95: +0.0147


## Deflated Sharpe Ratio (overfitting control)

In [6]:
from backtest.validation.deflated_sharpe import compute_dsr, probabilistic_sharpe_ratio

for n in [1, 10, 50, 200]:
    print(f"DSR (n_trials={n:>3}): {compute_dsr(rets, n_trials=n):.4f}")
print(f"PSR            : {probabilistic_sharpe_ratio(rets):.4f}")

DSR (n_trials=  1): 0.5756
DSR (n_trials= 10): 0.0832
DSR (n_trials= 50): 0.0185
DSR (n_trials=200): 0.0050
PSR            : 0.5756


## BHY multiple-testing correction across variants

In [7]:
from scipy import stats

from backtest.validation.multiple_testing import MultipleTestingReport, bhy_correction

variants = {"mom_63": 63, "mom_126": 126, "mom_252": 252}
pvals, labels = [], []
for label, lb in variants.items():
    s = MultiAssetRotation(mom_lookback=lb)
    w = s.generate_weights(prices, weekly)
    r = VectorizedBacktest(TransactionCostModel()).run(w, prices).returns.dropna()
    t = r.mean() / (r.std(ddof=1) + 1e-12) * np.sqrt(len(r))
    pvals.append(float(2 * (1 - stats.norm.cdf(abs(t))))); labels.append(label)
reject, adj = bhy_correction(pvals, alpha=0.05)
print(MultipleTestingReport(pvals, labels=labels).summary())

Multiple-testing report (3 tests, alpha=0.05)
--------------------------------------------------------
  Raw significant (no correction): 0
  Benjamini-Hochberg (FDR):        0
  Benjamini-Hochberg-Yekutieli:    0
  Bonferroni (FWER):               0
--------------------------------------------------------
  Note: factor mining over many candidates inflates false
  positives; prefer BHY when strategy returns are correlated.


In [8]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
result.equity_curve.plot(ax=ax[0]); ax[0].set_title("Strategy equity"); ax[0].set_ylabel("NAV")
dd = ts.drawdown_series()
dd.plot(ax=ax[1], color="crimson"); ax[1].fill_between(dd.index, dd.values, 0, color="crimson", alpha=0.3)
ax[1].set_title("Drawdown"); plt.tight_layout(); print("backtest analysis complete.")

backtest analysis complete.
